# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsf-rawnak/FlyRankAI-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Approach:** I follow `scripts/ml_utils.py`'s existing `MODEL_NUMERIC_FEATURES` /
`MODEL_CATEGORICAL_FEATURES` lists rather than inventing my own — they already encode the two
rules that matter (no identifiers, no label-source columns). Numerics get zero-filled per the
data dictionary's missingness note, and the four heavy-tailed traffic totals
(`impressions_90d`, `clicks_90d`, `sessions_90d`, `ai_sessions_90d`) get `log1p`'d before they
become features, since a raw traffic count spans 5+ orders of magnitude and would dominate any
linear model untransformed.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

local_path = Path("../../data/raw/content_refresh_anonymized.csv")
raw_url = "https://raw.githubusercontent.com/rsf-rawnak/FlyRankAI-ML-Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(local_path) if local_path.exists() else pd.read_csv(raw_url)

MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
    "word_count_tier", "impression_tier", "position_tier",
]

numeric_fill_zero = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
for col in numeric_fill_zero:
    df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

for col in ["competition_level", "content_type", "main_intent", "age_tier",
            "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]:
    df[col] = df[col].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

feature_df = df[["content_id", "client_id"] + MODEL_NUMERIC_FEATURES
                 + MODEL_CATEGORICAL_FEATURES + ["is_declining_label"]].copy()

print(f"Feature vector shape: {feature_df.shape}")
print(f"Missing values remaining after fill: {feature_df.isna().sum().sum()}")

# Data dictionary warns missingness is systematic by content_type, not random - confirm it
miss_by_type = df.groupby("content_type")[["search_volume", "competition", "cpc"]].apply(lambda g: (g == 0).mean())
print("\nShare of rows with 0-filled keyword columns, by content_type:")
print(miss_by_type.round(3))

Feature vector shape: (30000, 29)
Missing values remaining after fill: 0

Share of rows with 0-filled keyword columns, by content_type:
                    search_volume  competition    cpc
content_type                                         
comparison article          1.000         1.00  1.000
feedly article              1.000         1.00  1.000
keyword article             0.395         0.59  0.748


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE
the moment you predict.*

**Availability check first, because it matters more than it looks:** the task (from ML-03) is
"which pages should a reviewer look at first *this week*" — so every feature has to be something
known at the start of the review window, not something that only exists once the outcome is
already visible. All 18 numeric + 8 categorical features in section 1 pass that test: they're
built from the trailing-90-day window that ends *before* the label's 30-day comparison window
starts, not from it. The two columns that would fail it — `trend_pct` and `trend_direction` — are
excluded already; that's section 3.

**Missingness is real, not random** (confirmed above): `search_volume`, `competition`, and `cpc`
are 100% blank for both `comparison article` and `feedly article` rows — only `keyword article`
rows carry keyword-context data at all (and even those are ~40-75% populated). A blind
`fillna(0)` would silently teach a model "no keyword data" = "content_type is comparison/feedly",
which is why `content_type` stays in as its own categorical feature rather than letting the
zero-fills stand in for it implicitly.

**Categorical vs numeric split:** `competition_level`, `content_type`, `main_intent`,
`age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier` are all
threshold-derived buckets from the data dictionary — one-hot friendly, and useful because they
expose non-linear cutoffs a linear model can't find on its own in the raw numeric column.

In [2]:
# One row per model feature: what it means (short) and whether it's available before the
# review decision point (all should be True - that's the point of this check).
feature_notes = pd.DataFrame([
    ("search_volume", "keyword demand estimate", "0 when no keyword data", True),
    ("competition", "keyword competition 0-1", "0 when no keyword data", True),
    ("cpc", "cost-per-click estimate", "0 when no keyword data", True),
    ("word_count", "article length", "0 when not measured (7,699 rows)", True),
    ("char_count", "article character length", "0 when not measured", True),
    ("log_impressions_90d", "log1p of trailing 90d impressions", "no blanks", True),
    ("log_clicks_90d", "log1p of trailing 90d clicks", "no blanks", True),
    ("log_sessions_90d", "log1p of trailing 90d sessions", "no blanks", True),
    ("log_ai_sessions_90d", "log1p of AI-referred sessions", "no blanks", True),
    ("ctr", "clicks/impressions x100", "0 when impressions_90d=0 (not in this slice)", True),
    ("avg_position", "mean GSC position", "0 = no position data, not position zero", True),
    ("engagement_rate", "engaged/total sessions x100", "0 when 0 engaged sessions", True),
    ("scroll_rate", "scroll events/pageviews x100", "blank->0 when pageviews=0", True),
    ("ai_traffic_pct", "AI-referred share of sessions", "0 when 0 AI sessions", True),
    ("content_age_days", "days since content created", "no blanks in this slice", True),
    ("days_since_last_update", "days since last edit", "no blanks (freshness_tier=never has 0 rows)", True),
], columns=["feature", "meaning", "missing_handling", "available_before_decision"])
feature_notes

,feature,meaning,missing_handling,available_before_decision
0,search_volume,keyword demand estimate,0 when no keyword data,True
1,competition,keyword competition 0-1,0 when no keyword data,True
2,cpc,cost-per-click estimate,0 when no keyword data,True
3,word_count,article length,"0 when not measured (7,699 rows)",True
4,char_count,article character length,0 when not measured,True
5,log_impressions_90d,log1p of trailing 90d impressions,no blanks,True
6,log_clicks_90d,log1p of trailing 90d clicks,no blanks,True
7,log_sessions_90d,log1p of trailing 90d sessions,no blanks,True
8,log_ai_sessions_90d,log1p of AI-referred sessions,no blanks,True
9,ctr,clicks/impressions x100,0 when impressions_90d=0 (not in this slice),True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**The smoking gun:** `is_declining_label` is defined as `trend_direction == "down"` — so
`trend_direction` isn't correlated with the label, it *is* the label wearing a different name.
The agreement check below confirms it's a perfect 1.0 match. `trend_pct` is the literal number
`trend_direction` is bucketed from (`>-20%` = down), so it's excluded for the same reason even
though its raw correlation with the label looks unremarkable — the risk isn't correlation
strength, it's that the formula for one *is* the other.

**A second, quieter leak risk: `client_id` and split design, not feature content.** It's excluded
as a feature (it's a pseudonym, not a signal), but the real danger is a *random* row-level
train/test split. Decline rates vary enormously by client — from 0% to 93.7% of a client's pages
— so if the same client's pages land in both train and test, the model can partly memorize
"client X's pages usually decline" instead of learning transferable signal. That's why the data
dictionary calls for **client-holdout** splits, not random ones.

In [3]:
# 1. trend_direction / trend_pct vs the label - identity check
agreement = ((df["trend_direction"] == "down").astype(int) == df["is_declining_label"]).mean()
print(f"Agreement between (trend_direction=='down') and is_declining_label: {agreement:.3f}  <- exact identity, not correlation")
print(f"corr(trend_pct, label) = {df['trend_pct'].fillna(0).corr(df['is_declining_label']):.3f}  <- looks mild, but it's the SOURCE the label buckets from")

# 2. client_id group-leakage risk for splitting (not a feature, but a split-design hazard)
client_rate = df.groupby("client_id")["is_declining_label"].mean()
print(f"\nDecline rate by client_id: min={client_rate.min():.3f}  max={client_rate.max():.3f}  std={client_rate.std():.3f}")
print("-> a random row split would leak client identity into the model; must group-split by client_id instead")

Agreement between (trend_direction=='down') and is_declining_label: 1.000  <- exact identity, not correlation
corr(trend_pct, label) = -0.131  <- looks mild, but it's the SOURCE the label buckets from

Decline rate by client_id: min=0.000  max=0.937  std=0.227
-> a random row split would leak client identity into the model; must group-split by client_id instead


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **`content_id`, `client_id`** — pseudonymous identifiers. Grouping/joins and split design
  only; using them as features would let the model memorize specific pages/clients instead of
  learning generalizable signal.
- **`trend_direction`, `trend_pct`** — the label's own definition. Section 3's agreement check
  (1.0) shows this isn't a strong correlation to be cautious about, it's outright identity —
  including either would let the model "predict" the label by reading it off a renamed copy of
  itself.
- **`provider_used`, `model_used`** — 71% and 19% blank respectively in this slice (checked
  below), and what they actually describe (which LLM authored the draft) isn't a *performance*
  signal a reviewer can act on — it doesn't tell you why a page is declining, just how it was
  first written. The data dictionary flags both as "not a model feature" for the same reason.

In [4]:
excluded = pd.DataFrame([
    ("content_id", "identifier", "grouping/joins only, not a signal"),
    ("client_id", "identifier", "grouping + client-holdout splits only"),
    ("trend_direction", "leakage", "IS the label (agreement=1.0, see section 3)"),
    ("trend_pct", "leakage", "the raw value trend_direction buckets from"),
    ("provider_used", "not predictive / mostly missing", f"{df['provider_used'].isna().mean():.1%} blank; describes authorship, not performance"),
    ("model_used", "not predictive / mostly missing", f"{(df['model_used'].isna() | (df['model_used']=='unknown')).mean():.1%} blank/unknown; same reason"),
], columns=["column", "reason", "detail"])
excluded

,column,reason,detail
0,content_id,identifier,"grouping/joins only, not a signal"
1,client_id,identifier,grouping + client-holdout splits only
2,trend_direction,leakage,"IS the label (agreement=1.0, see section 3)"
3,trend_pct,leakage,the raw value trend_direction buckets from
4,provider_used,not predictive / mostly missing,"71.5% blank; describes authorship, not perform..."
5,model_used,not predictive / mostly missing,21.6% blank/unknown; same reason


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.